# CS 4375 — Assignment 2, Question 5(c)
## Logistic Regression from Scratch using Gradient Descent

**Rules:** Implement from scratch using only `numpy` and `matplotlib`. No scikit-learn for the classifier.

**Dataset:** Iris — two features (sepal length, sepal width), binary classification (Versicolor vs Virginica).

**Goal:** Implement gradient ascent on the log-likelihood derived in Q5(b), plot the decision boundary, report test accuracy.

---

### Recap of the Math from Q5(a) and Q5(b)

**Hypothesis function** (sigmoid applied to linear combination):
$$p(Y=1 \mid x) = \sigma(w^Tx + b) = \frac{\exp(w^Tx + b)}{1 + \exp(w^Tx + b)}$$

**Log-likelihood** we maximise (derived in Q5b):
$$\ell(w,b) = \sum_{i=1}^{N} \frac{y^{(i)}+1}{2}(w^Tx^{(i)} + b) - \ln\left(1 + \exp(w^Tx^{(i)} + b)\right)$$

**Gradient ascent update rules** (from Prof. Iyer's slide 16):
$$\frac{\partial \ell}{\partial w_j} = \sum_{i=1}^{N} x_j^{(i)} \left[\frac{y^{(i)}+1}{2} - p(Y=1 \mid x^{(i)})\right]$$
$$\frac{\partial \ell}{\partial b} = \sum_{i=1}^{N} \left[\frac{y^{(i)}+1}{2} - p(Y=1 \mid x^{(i)})\right]$$

$$w_j^{t+1} \leftarrow w_j^t + \eta \cdot \frac{\partial \ell}{\partial w_j}, \qquad b^{t+1} \leftarrow b^t + \eta \cdot \frac{\partial \ell}{\partial b}$$

## Step 1: Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# sklearn used ONLY for loading Iris — not for classification
from sklearn.datasets import load_iris

print("Imports successful.")

## Step 2: Load and Prepare the Dataset

### Why These Specific Choices?

**Two features only (sepal length + sepal width):**  
Logistic Regression learns a linear decision boundary. With 2 features, that boundary is a straight line we can actually *plot* on a 2D graph — which is exactly what Q5(c) asks for. With 4 features the boundary would be a hyperplane in 4D space — impossible to visualise.

**Versicolor vs Virginica only (drop Setosa):**  
Logistic Regression as derived in Q5(b) is a **binary** classifier — it outputs one probability for class +1 vs class -1. Iris has 3 classes so we must pick 2. Versicolor vs Virginica is the harder, more interesting pair — Setosa is trivially separable from everything else.

**Labels as +1 / -1:**  
Our gradient formula from Q5(b) uses $y \in \{-1, +1\}$ — the $\frac{y+1}{2}$ indicator trick only works with this encoding. Versicolor → +1, Virginica → -1.

In [ ]:
# ── Load Iris ──────────────────────────────────────────────────────────────
iris = load_iris()

# Use only the first two features so we can plot the decision boundary in 2D
# Feature 0: sepal length (cm)
# Feature 1: sepal width  (cm)
all_features_2d = iris.data[:, :2]   # shape: (150, 2)
all_labels      = iris.target         # shape: (150,)  values: 0=Setosa, 1=Versicolor, 2=Virginica

# ── Keep only Versicolor (1) and Virginica (2) — drop Setosa ──────────────
# Boolean mask: True for the rows we want to keep
binary_class_mask   = (all_labels == 1) | (all_labels == 2)
X_binary            = all_features_2d[binary_class_mask]   # shape: (100, 2)
y_raw               = all_labels[binary_class_mask]        # shape: (100,)  values: 1 or 2

# ── Re-encode labels to +1 / -1 as required by Q5(b) gradient formulas ────
# Versicolor (original label 1) → +1
# Virginica  (original label 2) → -1
y_binary = np.where(y_raw == 1, 1, -1)   # shape: (100,)  values: +1 or -1

print(f"Dataset size   : {len(y_binary)} samples")
print(f"Feature shape  : {X_binary.shape}  (sepal length, sepal width)")
print(f"Label encoding : Versicolor=+1, Virginica=-1")
print(f"Class counts   : +1={np.sum(y_binary==1)}, -1={np.sum(y_binary==-1)}")

# ── 80/20 Train/Test Split (same approach as Q1) ──────────────────────────
np.random.seed(42)   # reproducible shuffle
shuffled_indices = np.random.permutation(len(y_binary))

split_point   = int(0.8 * len(y_binary))       # 80 train, 20 test
train_indices = shuffled_indices[:split_point]
test_indices  = shuffled_indices[split_point:]

X_train = X_binary[train_indices]   # shape: (80, 2)
y_train = y_binary[train_indices]   # shape: (80,)
X_test  = X_binary[test_indices]    # shape: (20, 2)
y_test  = y_binary[test_indices]    # shape: (20,)

print(f"\nTraining samples : {len(y_train)}")
print(f"Test samples     : {len(y_test)}")

## Step 3: Feature Normalisation

Gradient descent is sensitive to feature scale. If one feature ranges 0–10 and another ranges 0–10,000, the gradient steps are dominated by the large-scale feature and learning becomes slow or unstable. We apply **z-score normalisation** (standardisation) so both features have mean=0 and std=1.

$$x_{\text{normalised}} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

**Critical rule:** compute $\mu$ and $\sigma$ from the **training set only**, then apply those same values to the test set. Never fit on test data — that would be information leakage.

In [ ]:
# ── Compute mean and std FROM TRAINING DATA ONLY ──────────────────────────
# axis=0 means compute along rows → gives one mean/std per feature column
training_feature_means = X_train.mean(axis=0)   # shape: (2,)
training_feature_stds  = X_train.std(axis=0)    # shape: (2,)

# ── Apply the same normalisation to both train and test ───────────────────
# Using training stats on test set — intentional, not a bug
X_train_normalised = (X_train - training_feature_means) / training_feature_stds
X_test_normalised  = (X_test  - training_feature_means) / training_feature_stds

print(f"Training feature means : {training_feature_means}")
print(f"Training feature stds  : {training_feature_stds}")
print(f"\nAfter normalisation:")
print(f"  X_train mean ≈ {X_train_normalised.mean(axis=0).round(6)}  (should be ~0)")
print(f"  X_train std  ≈ {X_train_normalised.std(axis=0).round(6)}   (should be ~1)")

## Step 4: Implement Logistic Regression from Scratch

Three methods that directly implement the math from Q5(a) and Q5(b):

| Method | What it does | Where the math comes from |
|---|---|---|
| `sigmoid()` | Squashes any real number to (0,1) | Q5(a) hypothesis function |
| `fit()` | Runs gradient ascent to learn `w` and `b` | Q5(b) gradient update rules |
| `predict()` | Classifies new points using learned `w` and `b` | Q5(a) decision rule |

In [ ]:
class LogisticRegressionFromScratch:
    """
    Binary Logistic Regression implemented from scratch using gradient ascent.

    Uses labels y ∈ {-1, +1} as per Prof. Iyer's slides.
    Maximises the conditional log-likelihood derived in Q5(b).

    Parameters
    ----------
    learning_rate : float
        Step size η for each gradient ascent update. Too large → overshoots
        the maximum. Too small → converges very slowly.
    num_iterations : int
        Number of gradient ascent steps to take. More iterations → closer
        to the true maximum, but with diminishing returns.
    """

    def __init__(self, learning_rate, num_iterations):
        self.learning_rate  = learning_rate    # η in the update rule
        self.num_iterations = num_iterations
        self.weights        = None             # w — one value per feature
        self.bias           = None             # b — scalar intercept term
        self.log_likelihood_history = []       # track ℓ(w,b) over iterations

    # ── Sigmoid Function ──────────────────────────────────────────────────
    def _sigmoid(self, linear_combination):
        """
        Compute the sigmoid (logistic) function.

        σ(z) = exp(z) / (1 + exp(z)) = 1 / (1 + exp(-z))

        Maps any real number z to the open interval (0, 1).
        This is p(Y=1 | x, w, b) from Q5(a).

        We use np.clip to prevent overflow in exp() for very large
        negative z values — a numerical stability trick.
        """
        # Clip to prevent exp(-z) overflowing to infinity for very negative z
        clipped_input = np.clip(linear_combination, -500, 500)
        return 1.0 / (1.0 + np.exp(-clipped_input))

    # ── Log-Likelihood ────────────────────────────────────────────────────
    def _compute_log_likelihood(self, X, y, prob_of_positive_class):
        """
        Compute the log-likelihood ℓ(w,b) from Q5(b):

            ℓ = Σ [ (y+1)/2 · (wᵀx + b) - ln(1 + exp(wᵀx + b)) ]

        Used only for tracking convergence — not needed for the updates.
        """
        linear_combination = np.dot(X, self.weights) + self.bias
        # The indicator (y+1)/2 is 1 when y=+1, 0 when y=-1
        positive_class_indicator = (y + 1) / 2
        log_likelihood = np.sum(
            positive_class_indicator * linear_combination
            - np.log(1 + np.exp(np.clip(linear_combination, -500, 500)))
        )
        return log_likelihood

    # ── Training via Gradient Ascent ──────────────────────────────────────
    def fit(self, X_train, y_train):
        """
        Learn weights w and bias b by maximising the log-likelihood.

        Implements the gradient ascent update from Prof. Iyer's slide 16:

            ∂ℓ/∂w_j = Σ x_j^(i) · [ (y^(i)+1)/2 - p(Y=1 | x^(i)) ]
            ∂ℓ/∂b   = Σ           [ (y^(i)+1)/2 - p(Y=1 | x^(i)) ]

            w ← w + η · ∂ℓ/∂w
            b ← b + η · ∂ℓ/∂b

        The term [ (y+1)/2 - p(Y=1|x) ] is the PREDICTION ERROR:
            - (y+1)/2 converts our ±1 labels to 0/1 (true probability)
            - p(Y=1|x) is what the model predicted
            - Their difference is how wrong we were
        """
        num_samples, num_features = X_train.shape

        # Initialise weights and bias to zero — standard starting point
        self.weights = np.zeros(num_features)   # shape: (2,) for 2 features
        self.bias    = 0.0

        # ── Gradient Ascent Loop ──────────────────────────────────────────
        for iteration in range(self.num_iterations):

            # Step 1 — compute wᵀx + b for every training point at once
            # np.dot(X, w) gives a vector of N dot products simultaneously
            linear_combination = np.dot(X_train, self.weights) + self.bias
            # shape: (num_samples,)  e.g. [0.3, -1.2, 0.8, ...]

            # Step 2 — pass through sigmoid to get p(Y=1 | x^(i)) for all i
            # This is the model's current predicted probability for each point
            prob_positive_class = self._sigmoid(linear_combination)
            # shape: (num_samples,)  values between 0 and 1

            # Step 3 — convert ±1 labels to 0/1 indicator  (y+1)/2
            # This is the TRUE probability (0 or 1) for each training point
            true_class_indicator = (y_train + 1) / 2
            # shape: (num_samples,)  values: 0.0 or 1.0

            # Step 4 — prediction error for each training point
            # Positive error → model under-predicted → push w up
            # Negative error → model over-predicted  → push w down
            prediction_error = true_class_indicator - prob_positive_class
            # shape: (num_samples,)

            # Step 5 — compute gradients (vectorised over all N samples)
            # ∂ℓ/∂w_j = Σ x_j^(i) · error^(i)  for each feature j
            # np.dot(X.T, error) does this for ALL features at once
            gradient_weights = np.dot(X_train.T, prediction_error)
            # shape: (num_features,) — one gradient value per weight

            # ∂ℓ/∂b = Σ error^(i)  (no x term since b has no feature attached)
            gradient_bias = np.sum(prediction_error)
            # shape: scalar

            # Step 6 — gradient ASCENT update  (+ because we're maximising)
            self.weights += self.learning_rate * gradient_weights
            self.bias    += self.learning_rate * gradient_bias

            # Track log-likelihood every 100 iterations to monitor convergence
            if iteration % 100 == 0:
                current_log_likelihood = self._compute_log_likelihood(
                    X_train, y_train, prob_positive_class
                )
                self.log_likelihood_history.append(current_log_likelihood)

        return self

    # ── Prediction ────────────────────────────────────────────────────────
    def predict(self, X):
        """
        Classify new points using the learned weights and bias.

        Decision rule from Prof. Iyer's slide 6:
            Predict +1 if  wᵀx + b > 0  (sigmoid output > 0.5)
            Predict -1 if  wᵀx + b < 0  (sigmoid output < 0.5)

        Returns labels in {-1, +1} to match our encoding.
        """
        linear_combination = np.dot(X, self.weights) + self.bias
        prob_positive_class = self._sigmoid(linear_combination)

        # Convert probabilities to class labels: above 0.5 → +1, below → -1
        predicted_labels = np.where(prob_positive_class >= 0.5, 1, -1)
        return predicted_labels

    # ── Accuracy ──────────────────────────────────────────────────────────
    def score(self, X, y_true):
        """
        Compute classification accuracy: correct predictions / total predictions.
        """
        predicted_labels = self.predict(X)
        num_correct      = np.sum(predicted_labels == y_true)
        return num_correct / len(y_true)


print("LogisticRegressionFromScratch class defined successfully.")

## Step 5: Train the Model

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────
# learning_rate: how big each gradient ascent step is
#   Too large (e.g. 1.0)  → overshoots, oscillates, may diverge
#   Too small (e.g. 0.0001) → takes forever to converge
#   0.1 is a reasonable starting point for normalised features
LEARNING_RATE  = 0.1

# num_iterations: how many gradient steps to take
#   1000 is typically enough for a simple 2-feature binary problem
NUM_ITERATIONS = 1000

# ── Build and train ────────────────────────────────────────────────────────
logistic_model = LogisticRegressionFromScratch(
    learning_rate  = LEARNING_RATE,
    num_iterations = NUM_ITERATIONS
)

logistic_model.fit(X_train_normalised, y_train)

# ── Report learned parameters ──────────────────────────────────────────────
print(f"Training complete ({NUM_ITERATIONS} gradient ascent iterations)")
print(f"\nLearned parameters:")
print(f"  w (weights) = {logistic_model.weights}")
print(f"    w[0] = {logistic_model.weights[0]:.4f}  (sepal length weight)")
print(f"    w[1] = {logistic_model.weights[1]:.4f}  (sepal width weight)")
print(f"  b (bias)    = {logistic_model.bias:.4f}")
print(f"\nDecision boundary equation: {logistic_model.weights[0]:.4f}·x₁ + {logistic_model.weights[1]:.4f}·x₂ + ({logistic_model.bias:.4f}) = 0")

## Step 6: Evaluate on the Test Set

In [ ]:
# ── Accuracy ───────────────────────────────────────────────────────────────
train_accuracy = logistic_model.score(X_train_normalised, y_train)
test_accuracy  = logistic_model.score(X_test_normalised,  y_test)

print("=" * 50)
print("  Q5(c): Logistic Regression Results")
print("=" * 50)
print(f"  Training accuracy : {train_accuracy*100:.1f}%")
print(f"  Test accuracy     : {test_accuracy*100:.1f}%")

# ── Per-class breakdown ────────────────────────────────────────────────────
test_predictions = logistic_model.predict(X_test_normalised)
print(f"\nPer-class breakdown on test set:")

for label, name in [(1, 'Versicolor (+1)'), (-1, 'Virginica  (-1)')]:
    true_mask      = (y_test == label)
    num_correct    = np.sum(test_predictions[true_mask] == label)
    num_total      = np.sum(true_mask)
    print(f"  {name}: {num_correct}/{num_total} correct")

## Step 7: Plot the Decision Boundary

### How the Decision Boundary Plot Works

The decision boundary is the line where $w^Tx + b = 0$ — the exact threshold between predicting +1 and -1. We visualise it by:

1. Creating a dense **meshgrid** of points covering the entire feature space
2. Running `predict()` on every single point in that grid
3. Colouring each point by its predicted class

The colour boundary that emerges IS the decision boundary — the line the model has learned to separate Versicolor from Virginica.

In [ ]:
# ── Build the meshgrid ─────────────────────────────────────────────────────
# We need the range of the normalised features to set grid boundaries
x_axis_min = X_train_normalised[:, 0].min() - 0.5   # sepal length axis
x_axis_max = X_train_normalised[:, 0].max() + 0.5
y_axis_min = X_train_normalised[:, 1].min() - 0.5   # sepal width axis
y_axis_max = X_train_normalised[:, 1].max() + 0.5

# Create two 1D arrays of evenly spaced values across each axis
# step=0.02 means the grid has a point every 0.02 units — fine enough to
# look smooth but not so fine it becomes slow
grid_step = 0.02
x_axis_grid_values = np.arange(x_axis_min, x_axis_max, grid_step)
y_axis_grid_values = np.arange(y_axis_min, y_axis_max, grid_step)

# np.meshgrid turns two 1D arrays into two 2D arrays covering every
# combination of (x, y) values — like a coordinate grid on graph paper
grid_x, grid_y = np.meshgrid(x_axis_grid_values, y_axis_grid_values)

# Flatten and stack into (n_grid_points, 2) so we can pass it to predict()
# grid_x.ravel() and grid_y.ravel() flatten the 2D grids into 1D arrays
# np.c_ stacks them as two columns: each row is one (x, y) grid point
all_grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]

# Predict the class for every grid point using our trained model
grid_predictions = logistic_model.predict(all_grid_points)

# Reshape predictions back to the 2D grid shape for contourf
grid_predictions_2d = grid_predictions.reshape(grid_x.shape)

# ── Plot ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

# contourf fills regions of the grid with colour based on predicted class
# alpha=0.3 makes it transparent so data points show clearly on top
ax.contourf(
    grid_x, grid_y, grid_predictions_2d,
    alpha=0.3,
    cmap=plt.cm.RdYlBu    # Red=Virginica(-1), Blue=Versicolor(+1)
)

# Draw the decision boundary line itself (where prediction switches class)
ax.contour(
    grid_x, grid_y, grid_predictions_2d,
    colors='black', linewidths=1.5, linestyles='--'
)

# Plot TRAINING points
versicolor_train_mask = (y_train == 1)
virginica_train_mask  = (y_train == -1)

ax.scatter(
    X_train_normalised[versicolor_train_mask, 0],
    X_train_normalised[versicolor_train_mask, 1],
    c='steelblue', marker='o', edgecolors='black',
    s=60, label='Versicolor +1 (train)', zorder=3
)
ax.scatter(
    X_train_normalised[virginica_train_mask, 0],
    X_train_normalised[virginica_train_mask, 1],
    c='tomato', marker='o', edgecolors='black',
    s=60, label='Virginica -1 (train)', zorder=3
)

# Plot TEST points with star markers so they stand out from training points
versicolor_test_mask = (y_test == 1)
virginica_test_mask  = (y_test == -1)

ax.scatter(
    X_test_normalised[versicolor_test_mask, 0],
    X_test_normalised[versicolor_test_mask, 1],
    c='blue', marker='*', edgecolors='black',
    s=200, label='Versicolor +1 (test)', zorder=4
)
ax.scatter(
    X_test_normalised[virginica_test_mask, 0],
    X_test_normalised[virginica_test_mask, 1],
    c='red', marker='*', edgecolors='black',
    s=200, label='Virginica -1 (test)', zorder=4
)

ax.set_title(
    f'Logistic Regression Decision Boundary\n'
    f'Test Accuracy = {test_accuracy*100:.1f}%  |  '
    f'η = {LEARNING_RATE}  |  Iterations = {NUM_ITERATIONS}',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Sepal Length (normalised)', fontsize=12)
ax.set_ylabel('Sepal Width (normalised)',  fontsize=12)
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('logistic_regression_decision_boundary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as logistic_regression_decision_boundary.png")

## Step 8: Log-Likelihood Convergence Plot

This shows the log-likelihood $\ell(w,b)$ increasing over training iterations — confirmation that gradient ascent is actually working and finding a better solution over time.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

iterations_recorded = np.arange(len(logistic_model.log_likelihood_history)) * 100

ax.plot(
    iterations_recorded,
    logistic_model.log_likelihood_history,
    color='steelblue', linewidth=2
)

ax.set_title('Log-Likelihood During Training\n(should increase and plateau — confirms gradient ascent is working)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Iteration', fontsize=11)
ax.set_ylabel('Log-Likelihood ℓ(w,b)', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('logistic_regression_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Convergence plot saved as logistic_regression_convergence.png")

## Step 9: Results Summary & Commentary

In [ ]:
print("=" * 55)
print("  RESULTS SUMMARY")
print("=" * 55)
print(f"  Dataset    : Iris — Versicolor vs Virginica")
print(f"  Features   : Sepal length, Sepal width (normalised)")
print(f"  Labels     : Versicolor=+1, Virginica=-1")
print(f"  Split      : 80 train / 20 test")
print(f"  η (lr)     : {LEARNING_RATE}")
print(f"  Iterations : {NUM_ITERATIONS}")
print()
print(f"  Learned weights : w = {logistic_model.weights.round(4)}")
print(f"  Learned bias    : b = {logistic_model.bias:.4f}")
print()
print(f"  Training accuracy : {train_accuracy*100:.1f}%")
print(f"  Test accuracy     : {test_accuracy*100:.1f}%")

print("""
── Commentary ────────────────────────────────────────────

The logistic regression model learns a linear decision
boundary by iteratively adjusting weights w and bias b
via gradient ascent on the log-likelihood derived in Q5(b).

Versicolor and Virginica overlap significantly when using
only sepal length and sepal width — these two features
are less discriminative than petal features. This is
reflected in the test accuracy, where some misclassifications
occur in the overlapping region near the decision boundary.

The convergence plot confirms gradient ascent is working:
log-likelihood increases with each iteration and plateaus
as the model approaches the maximum. This satisfies the
stopping criterion described in Prof. Iyer's slides.

Because logistic regression is a discriminative model
(it learns p(y|x) directly), it does not make the naive
independence assumption that Naive Bayes requires —
the features can be correlated without affecting validity.
""")